In [30]:
using Revise
using PVlib

In [31]:
@which panel_tilt_azimuth_3dof(position, pv_model.installation_tilt_deg, pv_model.installation_azimuth_deg, pv_model.initial_yaw_deg)

panel_tilt_azimuth_3dof(motion::AbstractVector{<:Real}, install_tilt_deg::Real, install_azimuth_deg::Real, yaw_offset_deg::Real)
     @ PVlib c:\Users\jtgrasb\Documents\GitHub\PVlib.jl\src\floating_utilities.jl:154

In [32]:
# solar wrapper functions

Base.@kwdef struct PvlibSolarModel{TM,TI,T<:Real}
    pv_module::TM
    inverter::Union{Nothing,TI} = nothing
    latitude::T
    longitude::T
    altitude_m::T
    installation_tilt_deg::T
    installation_azimuth_deg::T
    initial_yaw_deg::T
    albedo::T = T(0.25)
    use_inverter_ac::Bool = false
end

function pvlib_solar_model(;
    module_name::AbstractString = "Canadian Solar CS5P-220M [ 2009]",
    inverter_name::AbstractString = "ABB: MICRO-0.25-I-OUTD-US-208 [208V]",
    latitude::Real,
    longitude::Real,
    altitude_m::Real = 0.0,
    installation_tilt_deg::Real = 0.0,
    installation_azimuth_deg::Real = 180.0,
    initial_yaw_deg::Real = 0.0,
    albedo::Real = 0.25,
    use_inverter_ac::Bool = false,
    T::Type{<:Real} = Float64,
)
    module_params = PVlib.read_solar_module(module_name)
    inverter_params = PVlib.read_solar_inverter(inverter_name)

    return PvlibSolarModel{
        typeof(module_params), typeof(inverter_params), T
    }(
        module_params,
        inverter_params,
        convert(T, latitude), 
        convert(T, longitude), 
        convert(T, altitude_m), 
        convert(T, installation_tilt_deg),
        convert(T, installation_azimuth_deg),
        convert(T, initial_yaw_deg),
        convert(T, albedo),
        use_inverter_ac,
    )
end

# Takes in time, position, velocity, and control states.
# Returns force and power
function pvlib_wrapper_3dof(
    pv_model::PvlibSolarModel,
    weather_data::WeatherSample,
    solar_position::SolarPosition,
    t::Real,
    position::AbstractVector{<:Real},
    velocity::AbstractVector{<:Real},
    control_states::AbstractVector{<:Real},
)
    @assert length(position) == 3 "position must have length 3"
    @assert length(velocity) == 3 "velocity must have length 3"

    instantaneous_tilt_deg, instantaneous_azimuth_deg = panel_tilt_azimuth_3dof(position, pv_model.installation_tilt_deg, pv_model.installation_azimuth_deg, pv_model.initial_yaw_deg)
    
    total_irradiance = PVlib.get_total_irradiance(instantaneous_tilt_deg, instantaneous_azimuth_deg, weather_data, solar_position, pv_model.albedo)
    cell_temp = PVlib.sapm_cell_temperature(total_irradiance, weather_data)
    effective_irradiance = PVlib.sapm_effective_irradiance(total_irradiance, pv_model.pv_module, solar_position, pv_model.installation_tilt_deg, pv_model.installation_azimuth_deg, pv_model.altitude_m)
    dc_power = PVlib.sapm_dc_components(pv_model.pv_module, effective_irradiance, cell_temp)

    force_on_platform = zeros(promote_type(eltype(position), eltype(velocity)), 3)
    power = dc_power.p_mp

    return force, power
end

pvlib_wrapper_3dof (generic function with 2 methods)

In [42]:
# WEC functions

# Takes in time, position, velocity, and control states.
# Returns force and power
function wec_wrapper_3dof(
    pto_kinematics::AbstractVector{<:Real},
    t::Real,
    position::AbstractVector{<:Real},
    velocity::AbstractVector{<:Real},
    control_states::AbstractVector{<:Real},
)
    @assert length(position) == 3 "position must have length 3"
    @assert length(velocity) == 3 "velocity must have length 3"

    pto_damping = control_states[1]

    v_pto = pto_kinematics' * velocity
    f_pto = -pto_damping * v_pto

    force_on_platform = pto_kinematics * f_pto
    power = pto_damping * v_pto^2

    return force_on_platform, power
end

wec_wrapper_3dof (generic function with 3 methods)

In [43]:
latitude = 35.95
longitude = -75.125
altitude = 0.0
api_key = "E52b7mqeTWLigj2xF5Bn4n6Mm87ecm5LFFeYh4US" # from NLR
email = "jtgrasb@sandia.gov"
start_monthday=(1, 1)
end_monthday=(1, 1)

pv_model = pvlib_solar_model(
    latitude = latitude,
    longitude = longitude,
    altitude_m = altitude,
)

weather_data = get_meteorological_data_nsrdb(latitude, longitude, api_key, email, start_monthday, end_monthday, false)
weather_data = weather_data[12] # just use the 12th hour for this simulation
solar_position = get_solar_position(latitude, longitude, altitude, weather_data)

# WEC
pto_kinematics = [0, 0, 1]

3-element Vector{Int64}:
 0
 0
 1

In [44]:
t = 0.0
position = [0.1, 0.1, 0.2]
velocity = [0.1, 0.1, 0.5]
control_states = [1]

force_solar, power_solar = pvlib_wrapper_3dof(pv_model, weather_data, solar_position, t, position, velocity, control_states)

force_WEC, power_WEC = wec_wrapper_3dof(pto_kinematics, t, position, velocity, control_states)

println("Solar force = ", force_solar)
println("Solar power = ", power_solar)

println("WEC force = ", force_WEC)
println("WEC power = ", power_WEC)

Solar force = [0.0, 0.0, 0.0]
Solar power = 49.31537410780736
WEC force = [-0.0, -0.0, -0.5]
WEC power = 0.25
